# 2. B2·B4 병합 데이터 최종 전처리

| 항목 | 내용 |
|------|------|
| **입력** | `병합_데이터셋/df_B2_B4_merged.parquet` (~1.48 GB) |
| **출력** | `전처리_EDA/최종/df_전처리완료.parquet` |
| **분리** | `전처리_EDA/최종/df_이상치_검토용.parquet` |
| **리포트** | `전처리_EDA/최종/결측치_요약.csv` |

### 처리 단계
1. 컬럼명 변경 (영문 → 한글)
2. 시간 파생 변수 생성 (`판매시간_dt`, `판매월/주/일/요일/시간대`)
3. 거래 고유키 생성 (`점포코드_POS번호_거래번호`)
4. 결측치 누적 집계 (메모리 효율적 방식)
5. 이상치 플래그 추가 (삭제 없이 `flag_이상치_유형` 컬럼으로 분류)

> ⚠️ **메모리 주의:** PyArrow `iter_batches` 방식으로 200만 행 단위 처리합니다.  
> 이상치는 임의 삭제 없이 플래그 처리하며, 최종 의사결정은 사용자가 내립니다.

## 0. 라이브러리 임포트

In [1]:
import os
import gc
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

warnings.filterwarnings("ignore")
print("✅ 라이브러리 임포트 완료")

✅ 라이브러리 임포트 완료


## 1. 경로 설정 및 출력 디렉토리 생성

In [2]:
BASE_DIR   = r"C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터"
INPUT_PATH = os.path.join(BASE_DIR, "병합_데이터셋",  "df_B2_B4_merged.parquet")
OUTPUT_DIR = os.path.join(BASE_DIR, "전처리_EDA", "최종")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 출력 파일 경로
OUTPUT_MAIN    = os.path.join(OUTPUT_DIR, "df_전처리완료.parquet")      # 전체 데이터
OUTPUT_REVIEW  = os.path.join(OUTPUT_DIR, "df_이상치_검토용.parquet")   # 이상치 분리
OUTPUT_MISSING = os.path.join(OUTPUT_DIR, "결측치_요약.csv")            # 결측치 리포트

print(f"📂 입력 경로  : {INPUT_PATH}")
print(f"💾 출력 디렉토리: {OUTPUT_DIR}")

📂 입력 경로  : C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\병합_데이터셋\df_B2_B4_merged.parquet
💾 출력 디렉토리: C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\전처리_EDA\최종


## 2. 컬럼명 매핑 딕셔너리 정의

원본 영문 컬럼명 → 한글 컬럼명.  
향후 추가 컬럼이 확인되면 `RENAME_MAP` 하단 주석을 해제하여 적용하세요.

In [11]:
RENAME_MAP = {
    "SALE_DATE"      : "영업일자",
    "SALE_TIME"      : "판매시분초",
    "STORE_CODE"     : "점포코드",
    "POS_NO"         : "POS번호",
    "TRADE_NO"       : "거래번호",
    "ITEM_CODE"      : "상품코드",
    "SALE_QTY"       : "매출수량",
    "SALE_AMT"       : "매출금액",
    "ITEM_NM"        : "상품명",
    "ITEM_LRDV_NM"   : "상품대분류명",
    # ↓ 병합 데이터에 존재하는 경우 주석 해제
    "ITEM_MDDV_NM" : "상품중분류명",
    "ITEM_SMDV_NM" : "상품소분류명",
    # "PROMO_YN"     : "프로모션여부",
}

print(f"✅ 컬럼 매핑 정의 완료: {len(RENAME_MAP)}개")

✅ 컬럼 매핑 정의 완료: 12개


## 3. 파생변수 생성 헬퍼 함수 정의

### 3-1. 판매 날짜+시간 결합 → `datetime` 변환

In [4]:
def _make_datetime_col(df: pd.DataFrame) -> pd.Series:
    """
    '영업일자'(YYYYMMDD)와 '판매시분초'(HHMMSS)를 결합 → datetime Series 반환.

    처리 전략:
      - 정수형/문자열 모두 str 변환 후 결합
      - '판매시분초' 없으면 '000000' 대체
      - hour ≥ 24 등 비정상 값은 errors='coerce' 로 NaT 처리
    """
    date_str = df["영업일자"].astype(str).str.strip().str[:8]  # 앞 8자리
    if "판매시분초" in df.columns:
        time_str = (
            df["판매시분초"].astype(str).str.strip()
                          .str.zfill(6)   # 6자리 미만이면 앞에 0 채움
                          .str[:6]        # 앞 6자리만
        )
    else:
        time_str = pd.Series(["000000"] * len(df), index=df.index)

    combined = date_str + time_str        # 'YYYYMMDDHHMMSS'
    return pd.to_datetime(combined, format="%Y%m%d%H%M%S", errors="coerce")

### 3-2. 시간 파생 변수 6종 추가

In [5]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    판매시간_dt (datetime64) 생성 후 파생 컬럼 추가.

    생성 컬럼:
      판매시간_dt / 판매월 / 판매주 / 판매일 / 판매요일 / 판매시간대
    """
    df["판매시간_dt"] = _make_datetime_col(df)

    df["판매월"]     = df["판매시간_dt"].dt.month.astype("Int8")
    df["판매주"]     = df["판매시간_dt"].dt.isocalendar().week.astype("Int8")
    df["판매일"]     = df["판매시간_dt"].dt.day.astype("Int8")
    df["판매시간대"] = df["판매시간_dt"].dt.hour.astype("Int8")

    day_map = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}
    df["판매요일"]   = df["판매시간_dt"].dt.dayofweek.map(day_map).astype("category")

    return df

### 3-3. 거래 고유키 생성

In [6]:
def add_trade_key(df: pd.DataFrame) -> pd.DataFrame:
    """
    '점포코드_POS번호_거래번호' 형태의 거래_고유키 컬럼 생성.
    정수형으로 저장된 경우에도 str 변환 후 결합.
    """
    df["거래_고유키"] = (
        df["점포코드"].astype(str) + "_" +
        df["POS번호"].astype(str)  + "_" +
        df["거래번호"].astype(str)
    )
    return df

### 3-4. 이상치 플래그 추가

| 플래그 값 | 조건 | 비즈니스 해석 |
|-----------|------|---------------|
| `NORMAL` | 수량>0 AND 금액>0 | 정상 거래 |
| `REFUND_NEGATIVE` | 수량<0 AND 금액<0 | 반품/환불 |
| `ZERO_AMT_NEG_QTY` | 수량<0 AND 금액=0 | 재고 조정/손실 처리 |
| `DATA_SUSPECT` | 수량>0 AND 금액<0 | ⚠️ 시스템 오류 의심 |
| `PARTIAL_CANCEL` | 수량<0 AND 금액>0 | 부분 취소/할인 조정 |
| `GIFT_OR_ZERO_QTY` | 수량=0 OR 금액=0 | 증정품/샘플 가능성 |

> 🔴 **주의:** 이 셀은 플래그를 **추가**할 뿐이며, 데이터를 삭제하지 않습니다.

In [7]:
def add_anomaly_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    매출수량·매출금액 기준으로 flag_이상치_유형 컬럼 추가.
    데이터 삭제/상계 없이 분류만 수행 → 추후 사용자가 판단.
    """
    if "매출수량" not in df.columns or "매출금액" not in df.columns:
        df["flag_이상치_유형"] = "UNKNOWN_MISSING_COLS"
        return df

    qty = df["매출수량"]
    amt = df["매출금액"]

    conditions = [
        (qty > 0) & (amt > 0),                             # 정상
        (qty < 0) & (amt < 0),                             # 반품/환불
        (qty < 0) & (amt == 0),                            # 재고 조정
        (qty > 0) & (amt < 0),                             # 시스템 오류 의심
        ((qty < 0) & (amt > 0)) | ((qty > 0) & (amt < 0)),# 부분 취소
        (qty == 0) | (amt == 0),                           # 증정품
    ]
    choices = [
        "NORMAL",
        "REFUND_NEGATIVE",
        "ZERO_AMT_NEG_QTY",
        "DATA_SUSPECT",
        "PARTIAL_CANCEL",
        "GIFT_OR_ZERO_QTY",
    ]

    df["flag_이상치_유형"] = np.select(conditions, choices, default="NORMAL")
    df["flag_이상치_유형"] = df["flag_이상치_유형"].astype("category")
    return df

## 4. 유틸리티 함수 정의

In [8]:
def accumulate_missing(missing_acc: dict, chunk_df: pd.DataFrame) -> dict:
    """
    청크 단위로 컬럼별 결측치 수를 누적 집계한다.
    전체 데이터를 메모리에 올리지 않고 결측치 통계를 산출하는 핵심 로직.
    """
    for col in chunk_df.columns:
        cnt = int(chunk_df[col].isna().sum())
        missing_acc[col] = missing_acc.get(col, 0) + cnt
    return missing_acc


def infer_output_schema(sample_df: pd.DataFrame) -> pa.Schema:
    """
    전처리된 첫 번째 청크를 기반으로 pyarrow 출력 스키마를 추론한다.
    이후 모든 청크에 동일한 스키마가 적용되어 Parquet 일관성을 보장.
    """
    return pa.Schema.from_pandas(sample_df, preserve_index=False)


def preprocess_chunk(
    df: pd.DataFrame,
    rename_map: dict,
    available_rename: dict,
) -> pd.DataFrame:
    """
    단일 청크에 모든 전처리 단계를 순서대로 적용한다:
      Step 1. 컬럼명 변경
      Step 2. 시간 파생 변수 생성
      Step 3. 거래 고유키 생성
      Step 4. 이상치 플래그 추가
    """
    df = df.rename(columns=available_rename)  # Step 1
    df = add_time_features(df)               # Step 2
    df = add_trade_key(df)                   # Step 3
    df = add_anomaly_flags(df)               # Step 4
    return df


print("✅ 모든 함수 정의 완료")

✅ 모든 함수 정의 완료


## 5. Parquet 파일 메타데이터 확인

데이터를 메모리에 올리지 않고 스키마와 크기만 확인합니다.

In [12]:
meta = pq.read_metadata(INPUT_PATH)
schema = pq.read_schema(INPUT_PATH)
file_size_gb  = os.path.getsize(INPUT_PATH) / 1e9
original_cols = schema.names

print(f"📂 입력 파일 크기 : {file_size_gb:.2f} GB")
print(f"📊 총 행 수       : {meta.num_rows:,}")
print(f"📦 Row Group 수   : {meta.num_row_groups}")
print(f"\n🏷️  원본 컬럼 ({len(original_cols)}개):")
for c in original_cols:
    print(f"   - {c}")

# 실제로 존재하는 컬럼만 rename 대상으로 필터링 (오류 방지)
available_rename = {k: v for k, v in RENAME_MAP.items() if k in original_cols}
missing_from_map = [c for c in original_cols if c not in RENAME_MAP]

print(f"\n✅ 컬럼명 변경 대상: {len(available_rename)}개")
if missing_from_map:
    print(f"⚠️  매핑 정의 없어 원본명 유지: {missing_from_map}")

📂 입력 파일 크기 : 1.48 GB
📊 총 행 수       : 101,399,895
📦 Row Group 수   : 102

🏷️  원본 컬럼 (12개):
   - SALE_DATE
   - SALE_TIME
   - STORE_CODE
   - POS_NO
   - TRADE_NO
   - ITEM_CODE
   - SALE_QTY
   - SALE_AMT
   - ITEM_NM
   - ITEM_LRDV_NM
   - ITEM_MDDV_NM
   - ITEM_SMDV_NM

✅ 컬럼명 변경 대상: 12개


## 6. 메인 전처리 루프

**PyArrow `iter_batches`** 방식으로 200만 행 단위 배치 처리:  
- `(e)` 전체 전처리 결과 → `df_전처리완료.parquet` 스트리밍 저장  
- `(f)` NORMAL 외 이상치 행 → `df_이상치_검토용.parquet` 분리 저장

> ⏳ 파일 크기에 따라 수 분~수십 분 소요될 수 있습니다.

In [13]:
BATCH_SIZE = 2_000_000  # 배치당 200만 행 (메모리 ~2GB 이내)

pf = pq.ParquetFile(INPUT_PATH)

total_rows      = 0
missing_acc     = {}   # 결측치 누적 집계 딕셔너리
flag_counts_acc = {}   # 이상치 플래그 분포 누적 집계

writer_main   = None   # 전체 데이터용 Writer
writer_review = None   # 이상치 검토용 Writer
output_schema = None   # 첫 청크에서 확정

batch_num = 0
print("=" * 65)
print("🚀 df_B2_B4_merged.parquet 메모리 효율적 전처리 시작")
print("=" * 65)
print(f"\n🔄 청크 단위 처리 시작 (batch_size={BATCH_SIZE:,})...\n")

for batch in pf.iter_batches(batch_size=BATCH_SIZE):
    batch_num += 1
    df_raw     = batch.to_pandas()
    chunk_size = len(df_raw)
    total_rows += chunk_size

    # ── (a) 전처리 적용 ──────────────────────────────────────────────────
    df_proc = preprocess_chunk(df_raw, RENAME_MAP, available_rename)
    del df_raw
    gc.collect()

    # ── (b) 결측치 누적 집계 ─────────────────────────────────────────────
    missing_acc = accumulate_missing(missing_acc, df_proc)

    # ── (c) 이상치 플래그 분포 누적 ─────────────────────────────────────
    if "flag_이상치_유형" in df_proc.columns:
        for k, v in df_proc["flag_이상치_유형"].value_counts().items():
            flag_counts_acc[k] = flag_counts_acc.get(k, 0) + int(v)

    # ── (d) 첫 번째 청크에서 스키마 확정 & Writer 초기화 ─────────────────
    if writer_main is None:
        output_schema = infer_output_schema(df_proc)
        writer_main   = pq.ParquetWriter(OUTPUT_MAIN,   output_schema, compression="snappy")
        writer_review = pq.ParquetWriter(OUTPUT_REVIEW, output_schema, compression="snappy")
        print(f"   ✅ 출력 스키마 확정 ({len(output_schema.names)}개 컬럼)")
        print(f"   ✅ Parquet Writer 초기화 완료\n")

    # ── (e) 전체 → 메인 파일 스트리밍 저장 ──────────────────────────────
    table_proc = pa.Table.from_pandas(df_proc, schema=output_schema, preserve_index=False)
    writer_main.write_table(table_proc)

    # ── (f) 이상치 행 → 검토용 파일 분리 저장 ────────────────────────────
    if "flag_이상치_유형" in df_proc.columns:
        df_review = df_proc[df_proc["flag_이상치_유형"] != "NORMAL"].copy()
        if len(df_review) > 0:
            writer_review.write_table(
                pa.Table.from_pandas(df_review, schema=output_schema, preserve_index=False)
            )
        del df_review

    del df_proc, table_proc
    gc.collect()

    if batch_num % 5 == 0 or chunk_size < BATCH_SIZE:
        print(f"   [Batch {batch_num:>3}]  누적 처리 행: {total_rows:>12,}")

# ── Writer 닫기 ───────────────────────────────────────────────────────────────
if writer_main:
    writer_main.close()
if writer_review:
    writer_review.close()

print(f"\n✅ 전처리 완료: 총 {total_rows:,}행 처리")

🚀 df_B2_B4_merged.parquet 메모리 효율적 전처리 시작

🔄 청크 단위 처리 시작 (batch_size=2,000,000)...

   ✅ 출력 스키마 확정 (20개 컬럼)
   ✅ Parquet Writer 초기화 완료

   [Batch   5]  누적 처리 행:   10,000,000
   [Batch  10]  누적 처리 행:   20,000,000
   [Batch  15]  누적 처리 행:   30,000,000
   [Batch  20]  누적 처리 행:   40,000,000
   [Batch  25]  누적 처리 행:   50,000,000
   [Batch  30]  누적 처리 행:   60,000,000
   [Batch  35]  누적 처리 행:   70,000,000
   [Batch  40]  누적 처리 행:   80,000,000
   [Batch  45]  누적 처리 행:   90,000,000
   [Batch  50]  누적 처리 행:  100,000,000
   [Batch  51]  누적 처리 행:  101,399,895

✅ 전처리 완료: 총 101,399,895행 처리


## 7. 결측치 요약 리포트 생성 및 저장

In [14]:
print("📋 결측치 요약 리포트 생성 중...")

if missing_acc:
    missing_df = pd.DataFrame.from_dict(
        missing_acc, orient="index", columns=["결측치_수"]
    )
    missing_df.index.name     = "컬럼명"
    missing_df["결측치_비율(%)"] = (missing_df["결측치_수"] / total_rows * 100).round(4)
    missing_df = missing_df.sort_values("결측치_수", ascending=False)
    missing_df.to_csv(OUTPUT_MISSING, encoding="utf-8-sig")

    print(f"\n💾 저장 완료: {OUTPUT_MISSING}")
    print("\n[결측치 현황 (상위 15개)]")
    display(missing_df.head(15).style.background_gradient(cmap="Reds", subset=["결측치_비율(%)"]))
else:
    print("   결측치 집계 결과 없음.")

📋 결측치 요약 리포트 생성 중...

💾 저장 완료: C:\Users\alexj\2026_캡스톤_디자인\세븐일레븐_프로젝트\세븐일레븐_내부데이터\전처리_EDA\최종\결측치_요약.csv

[결측치 현황 (상위 15개)]


,결측치_수,결측치_비율(%)
컬럼명,,
영업일자,0,0.000000
판매시분초,0,0.000000
거래_고유키,0,0.000000
판매요일,0,0.000000
판매시간대,0,0.000000
판매일,0,0.000000
판매주,0,0.000000
판매월,0,0.000000
판매시간_dt,0,0.000000


## 8. 이상치 플래그 분포 확인

In [15]:
print("📊 이상치 플래그 분포 (전체 데이터 기준):")
print("-" * 58)

flag_df = pd.DataFrame.from_dict(
    flag_counts_acc, orient="index", columns=["행_수"]
).sort_values("행_수", ascending=False)
flag_df["비율(%)"] = (flag_df["행_수"] / total_rows * 100).round(4)

for flag, row in flag_df.iterrows():
    print(f"  {flag:<25} : {row['행_수']:>12,}행  ({row['비율(%)']:>6.3f}%)")

print("-" * 58)
non_normal = flag_df.loc[flag_df.index != "NORMAL", "행_수"].sum()
print(f"  {'NORMAL 외 합계':<25} : {non_normal:>12,}행  ({non_normal/total_rows*100:>6.3f}%)")
print()

display(flag_df.style.background_gradient(cmap="YlOrRd", subset=["비율(%)"]))

📊 이상치 플래그 분포 (전체 데이터 기준):
----------------------------------------------------------
  NORMAL                    : 98,062,595.0행  (96.709%)
  REFUND_NEGATIVE           :  2,090,702.0행  ( 2.062%)
  GIFT_OR_ZERO_QTY          :  1,192,591.0행  ( 1.176%)
  ZERO_AMT_NEG_QTY          :     53,039.0행  ( 0.052%)
  DATA_SUSPECT              :        952.0행  ( 0.001%)
  PARTIAL_CANCEL            :         16.0행  ( 0.000%)
----------------------------------------------------------
  NORMAL 외 합계               :    3,337,300행  ( 3.291%)



,행_수,비율(%)
NORMAL,98062595,96.708800
REFUND_NEGATIVE,2090702,2.061800
GIFT_OR_ZERO_QTY,1192591,1.176100
ZERO_AMT_NEG_QTY,53039,0.052300
DATA_SUSPECT,952,0.000900
PARTIAL_CANCEL,16,0.000000


## 9. 최종 저장 파일 확인

In [16]:
print("=" * 65)
print("💾 저장 파일 목록")
print("=" * 65)

out_files = {
    "전처리 완료 (전체)": OUTPUT_MAIN,
    "이상치 검토용 분리": OUTPUT_REVIEW,
    "결측치 요약 CSV"   : OUTPUT_MISSING,
}
for label, path in out_files.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  ✅ [{label}]  →  {os.path.basename(path)}  ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ [{label}]  →  저장 실패 또는 이상치 없음")

print()
print("🎉 모든 전처리 단계 완료!")
print("=" * 65)
print()
print("📌 다음 단계 안내:")
print("   1. '결측치_요약.csv'를 확인하여 컬럼별 결측 비율을 검토하세요.")
print("   2. 'df_이상치_검토용.parquet'에서 flag_이상치_유형 별 처리 방향을")
print("      결정하세요. (삭제 / 유지 / 변환 등 — 사용자 최종 판단)")
print("   3. 'df_전처리완료.parquet'를 기반으로 후속 EDA를 진행하세요.")

💾 저장 파일 목록
  ✅ [전처리 완료 (전체)]  →  df_전처리완료.parquet  (2902.9 MB)
  ✅ [이상치 검토용 분리]  →  df_이상치_검토용.parquet  (130.3 MB)
  ✅ [결측치 요약 CSV]  →  결측치_요약.csv  (0.0 MB)

🎉 모든 전처리 단계 완료!

📌 다음 단계 안내:
   1. '결측치_요약.csv'를 확인하여 컬럼별 결측 비율을 검토하세요.
   2. 'df_이상치_검토용.parquet'에서 flag_이상치_유형 별 처리 방향을
      결정하세요. (삭제 / 유지 / 변환 등 — 사용자 최종 판단)
   3. 'df_전처리완료.parquet'를 기반으로 후속 EDA를 진행하세요.
